# 自然物と人工物の語は、どこで出会うか

Morris『*News from Nowhere*』とHudson『*A Crystal Age*』を、[発表スライド](https://cm3.github.io/dh-literature-20261016/)と同じ**固定116語**で数える実演です。「ランタイム」→「すべてのセルを実行」で始められます。最後のセルでは章や窓の長さを変えて試せます。

扱うのは名詞の原形だけです。章をまたがない50語窓を25語ずつ動かし、自然語と人工語が両方あれば「共存」と数えます。**共存は本文の意味を自動で判定した結果ではありません。**

講習中に多数の参加者がProject Gutenbergへ同時アクセスしないよう、Project Gutenbergの未加工TXTをこの[教材リポジトリ](https://github.com/cm3/dh-literature-20261016/tree/main/colab/data)から読み込みます。TXTにはライセンス全文が含まれます。出典：[Morris eBook #3261](https://www.gutenberg.org/ebooks/3261)、[Hudson eBook #7401](https://www.gutenberg.org/ebooks/7401)。Morrisの入力本文は1908年刊本に基づく転写で、作品の初出（1890年連載／1891年刊）と同一の版ではありません。


## 1. 本文と語彙表を読み込む

再現性のためNLTKの版を固定し、入力ファイルのSHA-256を確認します。初回だけNLTKの言語データ取得に少し時間がかかります。


In [ ]:
import csv
import hashlib
import importlib.metadata
import io
import os
import re
import subprocess
import sys
import unicodedata
import urllib.request
from bisect import bisect_right
from collections import Counter

NLTK_VERSION = "3.10.3"
try:
    installed = importlib.metadata.version("nltk")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != NLTK_VERSION:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"nltk=={NLTK_VERSION}"])

import nltk
from nltk.stem import WordNetLemmatizer

for lookup, package in {
    "tokenizers/punkt_tab/english": "punkt_tab",
    "taggers/averaged_perceptron_tagger_eng": "averaged_perceptron_tagger_eng",
    "corpora/wordnet": "wordnet",
}.items():
    try:
        nltk.data.find(lookup)
    except LookupError:
        if not nltk.download(package, quiet=True):
            raise RuntimeError(f"NLTKの言語データを取得できません: {package}")

BASE_URL = os.environ.get(
    "TALK_DEMO_BASE_URL",
    "https://raw.githubusercontent.com/cm3/dh-literature-20261016/main/",
).rstrip("/")
INPUTS = {
    "news_from_nowhere": (
        "colab/data/news_from_nowhere_pg3261.txt",
        "350beb623c7574515535a283adf7d2a78de4230c9b4b1e8f238b8b9a252661b1",
    ),
    "crystal_age": (
        "colab/data/crystal_age_pg7401.txt",
        "b414056b098b7c75a59d7e3ca00300ea06d0ac7457d88ef02a494a2fa91a3139",
    ),
    "lexicon": (
        "data/expanded_lexicon.csv",
        "6b61001c4e202f359d8a9b73175a107fd35605a89d99ed12855705dfb5d466e1",
    ),
}

def fetch_checked(path, expected_sha256):
    with urllib.request.urlopen(f"{BASE_URL}/{path}", timeout=45) as response:
        raw = response.read()
    actual = hashlib.sha256(raw).hexdigest()
    if actual != expected_sha256:
        raise ValueError(f"入力ファイルのハッシュが違います: {path} ({actual})")
    return raw

raw_inputs = {key: fetch_checked(*value) for key, value in INPUTS.items()}
print("本文2作品と116語の語彙表を確認しました。")


## 2. 小説本文だけを取り出す

未加工TXTに含まれる書誌情報・序文・巻末資料とライセンスは集計対象から外します。章見出しを統一し、章の開始位置を保存します。この処理はスライドの集計に用いた本文準備と同じ規則です。


In [ ]:
CONFIGS = {
    "news_from_nowhere": {
        "start": r"^CHAPTER I: DISCUSSION AND BED$",
        "start_occurrence": 1,
        "chapter": r"^CHAPTER ([IVXLCDM]+): (.+)$",
        "end": "FOOTNOTES:",
        "chapter_count": 32,
        "text_sha256": "b26f36b03135999202051056a14fb8076bc304525c999b91ad8b7f86a86b4179",
    },
    "crystal_age": {
        "start": r"^Chapter 1$",
        "start_occurrence": 1,
        "chapter": r"^Chapter (\d+)$",
        "end": "*** END OF THE PROJECT GUTENBERG EBOOK A CRYSTAL AGE ***",
        "chapter_count": 20,
        "text_sha256": "badd0de9ad2fbafb7a6b3aea2d283246675381b3d45c01265458c5d9ffb05827",
    },
}

def normalize(line):
    line = unicodedata.normalize("NFKC", line).replace("\u00a0", " ")
    return re.sub(r"\s+", " ", line).strip()

def int_to_roman(number):
    pairs = [(10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")]
    result = ""
    for value, numeral in pairs:
        while number >= value:
            result += numeral
            number -= value
    return result

def roman_number(value):
    values = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}
    return sum(
        -values[char] if i + 1 < len(value) and values[char] < values[value[i + 1]] else values[char]
        for i, char in enumerate(value)
    )

def prepare_text(raw, config):
    lines = raw.decode("utf-8-sig").splitlines()
    starts = [i for i, line in enumerate(lines) if re.fullmatch(config["start"], normalize(line))]
    ends = [i for i, line in enumerate(lines) if normalize(line) == config["end"]]
    if len(starts) != config["start_occurrence"] or len(ends) != 1 or ends[0] <= starts[-1]:
        raise ValueError(f"本文の開始・終了位置が予想と違います: {starts=}, {ends=}")

    blocks, paragraph, chapter_numbers = [], [], []
    def flush():
        if paragraph:
            blocks.append(" ".join(paragraph))
            paragraph.clear()

    for line in lines[starts[-1]:ends[0]]:
        line = normalize(line)
        heading = re.fullmatch(config["chapter"], line)
        if heading:
            flush()
            number = int(heading.group(1)) if heading.group(1).isdigit() else roman_number(heading.group(1))
            chapter_numbers.append(number)
            blocks.append(f"CHAPTER {int_to_roman(number)}.")
            if heading.lastindex == 2:
                blocks.append(heading.group(2).rstrip(".") + ".")
        elif line:
            paragraph.append(line)
        else:
            flush()
    flush()
    if chapter_numbers != list(range(1, config["chapter_count"] + 1)):
        raise ValueError(f"章番号が予想と違います: {chapter_numbers}")
    blocks.append("THE END.")
    text = "\n\n".join(blocks) + "\n"
    actual = hashlib.sha256(text.encode("utf-8")).hexdigest()
    if actual != config["text_sha256"]:
        raise ValueError(f"整形した本文のハッシュが予想と違います: {actual}")
    chapter_matches = list(re.finditer(r"(?m)^CHAPTER ([IVXLCDM]+)\.$", text))
    chapter_starts = [match.start() for match in chapter_matches]
    return text, chapter_starts

prepared = {key: prepare_text(raw_inputs[key], config) for key, config in CONFIGS.items()}
for key, (text, starts) in prepared.items():
    print(f"{key}: {len(starts)}章、{len(text):,}文字")


## 3. 名詞を原形にして116語と照合する

同じ綴りでも品詞が違う場合があります。ここではNLTKで `NN`・`NNS` と判定した名詞だけを、単数形の原形にして照合します。たとえば `houses` → `house` です。品詞判定も原形化も完全ではないので、具体的な用例は後で読み直します。


In [ ]:
lexicon_rows = list(csv.DictReader(io.StringIO(raw_inputs["lexicon"].decode("utf-8-sig"))))
nature = {row["lemma"] for row in lexicon_rows if row["semantic_class"] == "nature"}
built = {row["lemma"] for row in lexicon_rows if row["semantic_class"] == "built"}
assert len(nature) == 56 and len(built) == 60 and not nature & built
print(f"自然語 {len(nature)}語、人工語 {len(built)}語")
print("自然語の例:", ", ".join(sorted(nature)[:12]))
print("人工語の例:", ", ".join(sorted(built)[:12]))

TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")
lemmatizer = WordNetLemmatizer()

def tokenize_nouns(text, chapter_starts):
    sentences = [
        [token for token in nltk.word_tokenize(sentence) if TOKEN_RE.fullmatch(token)]
        for sentence in nltk.sent_tokenize(text)
    ]
    sentences = [sentence for sentence in sentences if sentence]
    records = []
    cursor = 0
    for batch_start in range(0, len(sentences), 500):
        tagged = nltk.pos_tag_sents(sentences[batch_start:batch_start + 500], lang="eng")
        for sentence in tagged:
            for surface, pos in sentence:
                char_start = text.find(surface, cursor)
                if char_start < 0:
                    raise ValueError(f"本文内で語を見つけられません: {surface!r}")
                cursor = char_start + len(surface)
                lemma = lemmatizer.lemmatize(surface.lower(), pos="n") if pos in {"NN", "NNS"} else ""
                records.append({
                    "surface": surface,
                    "lemma": lemma,
                    "chapter": max(1, bisect_right(chapter_starts, char_start)),
                    "char_start": char_start,
                    "char_end": cursor,
                })
    return records

tokens = {key: tokenize_nouns(*value) for key, value in prepared.items()}
for key, rows in tokens.items():
    print(f"{key}: {len(rows):,}トークン")


## 4. 50語窓を25語ずつ動かして数える

窓の中に両方の語があれば1件です。隣り合う窓は半分重なります。章の境界をまたぐ窓は作りません。


In [ ]:
def make_windows(rows, text, width=50, step=25):
    if width < 1 or step < 1:
        raise ValueError("widthとstepは正の整数にしてください")
    windows = []
    chapter_ranges = []
    for chapter in sorted({row["chapter"] for row in rows}):
        positions = [i for i, row in enumerate(rows) if row["chapter"] == chapter]
        chapter_ranges.append((chapter, positions[0], positions[-1] + 1))
    for chapter, first, last in chapter_ranges:
        for start in range(first, last - width + 1, step):
            group = rows[start:start + width]
            nature_hits = sorted({row["lemma"] for row in group if row["lemma"] in nature})
            built_hits = sorted({row["lemma"] for row in group if row["lemma"] in built})
            windows.append({
                "chapter": chapter,
                "start": start,
                "nature": nature_hits,
                "built": built_hits,
                "mixed": bool(nature_hits and built_hits),
                "excerpt": re.sub(r"\s+", " ", text[group[0]["char_start"]:group[-1]["char_end"]]).strip(),
            })
    return windows

windows = {key: make_windows(tokens[key], prepared[key][0]) for key in prepared}
expected = {"news_from_nowhere": (220, 3047), "crystal_age": (190, 2323)}
for key, work_windows in windows.items():
    mixed = sum(row["mixed"] for row in work_windows)
    total = len(work_windows)
    assert (mixed, total) == expected[key], f"スライドの値と異なります: {key}: {mixed}/{total}"
    print(f"{key}: {mixed:,}/{total:,}窓 = {100 * mixed / total:.2f}%")


## 5. 比較する・章ごとの違いを見る

この2作品ではHudsonの率が高くなります。ただし、これは固定116語と今回の窓の作り方における結果です。Morrisの内部でも章によって率が違います。


In [ ]:
import matplotlib.pyplot as plt

labels = {"news_from_nowhere": "Morris", "crystal_age": "Hudson"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
rates = [100 * sum(w["mixed"] for w in windows[key]) / len(windows[key]) for key in labels]
axes[0].bar(labels.values(), rates, color=["#297254", "#b4593c"])
axes[0].set_ylabel("Mixed windows (%)")
axes[0].set_ylim(0, 10)
axes[0].set_title("Two works")

by_chapter = []
for chapter in range(1, 33):
    group = [w for w in windows["news_from_nowhere"] if w["chapter"] == chapter]
    by_chapter.append(100 * sum(w["mixed"] for w in group) / len(group))
axes[1].bar(range(1, 33), by_chapter, color="#297254")
axes[1].set_xlabel("Chapter in News from Nowhere")
axes[1].set_ylabel("Mixed windows (%)")
axes[1].set_title("Variation within Morris")
plt.tight_layout()
plt.show()


## 6. 数字から本文へ戻る

章番号を変えて再実行できます。表示されるのは「両方の語を含む窓」の例です。自然と人工物の関係をどう読むかは、前後の本文も見て判断してください。


In [ ]:
selected_work = "news_from_nowhere"  # "crystal_age" にも変更できます
selected_chapter = 10                 # Morris: 1〜32 / Hudson: 1〜20

if selected_work not in windows:
    raise ValueError("selected_workが違います")
examples = [w for w in windows[selected_work]
            if w["chapter"] == selected_chapter and w["mixed"]]
print(f"{labels[selected_work]} 第{selected_chapter}章: 共存窓 {len(examples)}件")
for index, item in enumerate(examples[:3], start=1):
    print(f"\n【例{index}】自然: {', '.join(item['nature'])} / 人工: {', '.join(item['built'])}")
    print(item["excerpt"])


## 7. 分析の前提を変えてみる

50語窓を75語窓に変えると、Morrisで「共存」と判定される窓の数も、窓の総数も変わります。また `town` と `country` は本文に出ても、この116語の自然／人工分類には入っていません。文学史から都市／田園という軸を加えると、別の関係を問えます。


In [ ]:
longer = make_windows(tokens["news_from_nowhere"], prepared["news_from_nowhere"][0], width=75, step=25)
for width, group in [(50, windows["news_from_nowhere"]), (75, longer)]:
    count = sum(w["mixed"] for w in group)
    print(f"Morris {width}語窓: {count:,}/{len(group):,} = {100 * count / len(group):.2f}%")

assert "town" not in nature | built and "country" not in nature | built
counts = Counter(row["lemma"] for row in tokens["news_from_nowhere"])
print(f"本文の名詞: town {counts['town']}回、country {counts['country']}回（116語では未分類）")


### この実演の範囲

ここで再現するのはMorris・Hudsonの**固定116語による共存窓率**です。発表で紹介するBennettの13場面の人手判定（9/13）は、別の本文・語彙・判定法による分析で、このノートブックからは算出しません。数値の意味を考えるときは、[対訳と注釈](https://cm3.github.io/dh-literature-20261016/)にも戻ってみてください。
